# CLM-001-R3b: complete read-only posthoc analysis
Run All once. Fixed completed training 33734fdd1558; 102 tasks. Private HF is authoritative. No training, no Pilot, no Confirmation. Repeating Run All reuses verified results. A 6,900-second soft session budget is checked only between tasks; no task is killed.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
from kaggle_secrets import UserSecretsClient

EXPERIMENT = "CLM-001-R3b-POSTHOC"
ACTION = "full-analysis"  # All 102 fixed tasks, verified HF reuse, aggregates and publication.

SOURCE_SHA = "c29d2948610547d0cafa64acefb0a5d8a2e70577"
CODE = Path("/kaggle/working/minicells-r3b-analysis-code-" + SOURCE_SHA[:12])
WORK = Path("/kaggle/working/minicells-r3b-analysis-" + SOURCE_SHA[:12])
ENTRY = (
    CODE
    / "research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3b/analyze.py"
)
os.environ.update(
    OMP_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
    TOKENIZERS_PARALLELISM="false",
    HF_HUB_DISABLE_PROGRESS_BARS="1",
)


def execute(args, cwd=None):
    print("[R3b] " + " ".join(map(str, args)), flush=True)
    return subprocess.run(list(map(str, args)), cwd=cwd, check=True)


secrets = UserSecretsClient()
for name in (("HF_TOKEN", "GITHUB_TOKEN") if ACTION != "list-checkpoints" else ("HF_TOKEN",)):
    if not os.environ.get(name, "").strip():
        try:
            os.environ[name] = secrets.get_secret(name).strip()
        except Exception:
            raise RuntimeError("Enable existing Kaggle Secret " + name) from None
    if not os.environ[name]:
        raise RuntimeError("Missing existing Kaggle Secret " + name)

if not (CODE / ".git").exists():
    CODE.mkdir(parents=True, exist_ok=False)
    execute(["git", "init", "--quiet", CODE])
    execute(["git", "remote", "add", "origin", "https://github.com/ArcheLabs/mini-cells.git"], CODE)
    execute(["git", "sparse-checkout", "init", "--no-cone"], CODE)
    execute(
        [
            "git",
            "sparse-checkout",
            "set",
            "--no-cone",
            "/pyproject.toml",
            "/src/",
            "/shared/python/",
            "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/",
            "/research/projects/cellular-language-models/engineering/kaggle-continuation/",
            "/research/projects/cellular-language-models/experiments/006-path-first-clm/",
            "/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/",
            "/research/projects/continual-learning/context/04-continual-learning/scripts/research/",
            "/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/",
        ],
        CODE,
    )
    execute(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_SHA], CODE)
    execute(["git", "checkout", "--detach", "FETCH_HEAD"], CODE)
if subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CODE, text=True).strip() != SOURCE_SHA:
    raise RuntimeError("SOURCE_SHA_MISMATCH")

# Install only missing data/storage packages. Torch/numpy are supplied by Kaggle.
requirements = {
    "tokenizers": "tokenizers",
    "huggingface_hub": "huggingface_hub>=0.23",
    "matplotlib": "matplotlib",
}
missing = [
    requirement
    for module, requirement in requirements.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    execute([sys.executable, "-m", "pip", "install", "--quiet", *missing])
WORK.mkdir(parents=True, exist_ok=True)
print({"TRAINING_SOURCE_SHA": "a7792ef30582dfcaf324cca7e5245c8430df48b2",
       "PUBLIC_RUN_DIRECTORY": "33734fdd1558", "ANALYSIS_SOURCE_SHA": SOURCE_SHA,
       "planned_tasks": 102, "training_updates": 0, "confirmation": "SEALED"}, flush=True)
execute([sys.executable, ENTRY, "--mode", ACTION, "--output", WORK / "analysis",
         "--device", "cuda:0", "--session-seconds", "6900", "--publish-github"], CODE)
